In [ ]:
import random
from datasets import load_dataset
import pprint

# --- [데이터셋 정보 및 목표 설정] ---
# ✨ 이 데이터셋은 군사 관련 질문(question)과 정답(answer) 쌍으로 이루어져 있어요.
# ✨ 마치 군사 지식 테스트 문제집 같답니다!
# ✨ 목표: 이 구조화된 Q&A 데이터를 활용하여, 단순히 정답을 뽑아내는 것 이상의 '지능적인 프롬프트'를 생성하는 방법을 학습해 봅시다.
DATASET_NAME = "mulle0514/military"
TRAIN_SPLIT = "train"
SAMPLE_COUNT = 10 # 실습을 위해 상위 10개만 사용합니다.

# --- [함수 정의: 안전하게 데이터셋을 로드하는 마법의 함수] ---
def load_dataset_safely(dataset_name, split, sample_count):
    """
    스트리밍 모드를 시도하고, 실패할 경우 일반 모드로 안전하게 데이터를 로드하는 함수입니다.
    """.format()
    print("\n📚 데이터 로딩을 시작합니다... (지글지글)")
    
    # 1. 스트리밍 모드 시도 (가장 빠르고 효율적인 방법!)
    try:
        print("👉 [1단계] 스트리밍 모드로 로드 시도 (빠른 진행을 위해)...")
        dataset = load_dataset(dataset_name, split=split, streaming=True)
        
        # 🚨 스트리밍 데이터셋임을 확인하고, take() 패턴을 사용합니다.
        if hasattr(dataset, "take"):
            print("✨ 스트리밍 데이터셋(IterableDataset)입니다! .take() 패턴을 사용합니다.")
            # 샘플 전체를 리스트로 변환하여 사용하기 편리하게 만듭니다.
            sample_list = list(dataset.take(sample_count))
            return sample_list, "Stream (가상 메모리)", False
        else:
            # 이 경우는 거의 없지만, 만약을 대비합니다.
            print("⚠️ 경고: 스트리밍 모드였으나 take() 패턴을 찾을 수 없습니다.")
            return None, None, True

    # 2. 스트리밍 실패 시 예외 처리 및 일반 모드 사용
    except Exception as e:
        print(f"💥 스트리밍 로드 실패! 에러: {e}")
        print("✅ 대안으로, 작은 크기의 일반 데이터셋으로 로드합니다.")
        try:
            # 일반 Dataset 객체로 로드하여 안정적으로 진행합니다.
            dataset = load_dataset(dataset_name, split=split, streaming=False)
            print(f"✨ 일반 데이터셋(Dataset)으로 로드되었습니다. 총 {len(dataset)}개 샘플을 확인할 수 있어요.")
            return list(dataset.take(sample_count)), "Normal (실제 메모리)", True
        except Exception as e_fallback:
            print(f"😭 치명적인 데이터 로드 에러가 발생했습니다: {e_fallback}")
            return None, None, True


# =============================================================================
# 🧠 AI 튜터의 기발한 파이썬 실습 스크립트 시작!
# =============================================================================

# 안전한 데이터 로드를 실행하고 결과를 받습니다.
sample_data_list, load_type, is_fallback = load_dataset_safely(
    DATASET_NAME, TRAIN_SPLIT, SAMPLE_COUNT
)

if sample_data_list is None:
    print("\n😭 죄송해요! 데이터를 불러오는 데 문제가 생겨 실습을 진행할 수 없습니다. 데이터셋 ID를 확인해주세요.")
    exit()

print("\n" + "="*80)
print(f"🎉 성공! {load_type} 방식으로 데이터를 준비했어요! 총 {len(sample_data_list)}개 샘플로 실습을 시작합니다.")
print("="*80)

# -----------------------------------------------------------------------------
# 🧐 1. 데이터 탐색 및 기초 분석 (Quantitative Analysis)
# -----------------------------------------------------------------------------
print("\n✨ [Part 1] 데이터 탐색: 이 데이터가 어떤 모양인지 살펴봐요!")

total_question_len = 0
for sample in sample_data_list:
    question = sample['question']
    total_question_len += len(question)

avg_q_len = total_question_len / len(sample_data_list) if len(sample_data_list) > 0 else 0

print(f"🔎 분석 결과: 샘플 {len(sample_data_list)}개의 질문을 분석했더니...")
print(f"   ➡️ 평균 질문 길이: 약 {avg_q_len:.2f} 자입니다. (꽤 정보가 많은 질문들이네요!)")

# 첫 번째 샘플을 예로 보면서 데이터 구조를 익힙니다.
print("\n🔍 첫 번째 샘플 구조 확인:")
pprint.pprint(sample_data_list[0])


# -----------------------------------------------------------------------------
# 🤖 2. 창의적 AI 활용 실습: '지식 검증 프롬프트' 생성기
# -----------------------------------------------------------------------------
# 초보자용 실습 목표: 주어진 Q&A 쌍을 활용하여, LLM에게 "이 지식이 충분히 상세한지" 물어보는 프롬프트를 만들도록 코딩해 봅시다.
# 이 과정은 단순히 데이터를 보는 것을 넘어, AI가 어떻게 질문할지 '구조화'하는 능력을 길러줘요!

def generate_knowledge_check_prompt(question, answer, index):
    """
    주어진 Q&A 쌍을 활용하여, AI가 지식을 검증하는 프롬프트를 생성합니다.
    """
    # 템플릿을 사용합니다. (AI의 언어를 흉내 내는 재미있는 과정!)
    prompt = f"""
    [AI 지식 검증 요청]
    요청 지식 쌍:
    - 질문(Q): "{question}"
    - 답변(A): "{answer}"
    
    **당신의 임무:**
    위 Q&A 쌍이 너무 단순하거나 부족한 정보가 있는지 판단해주세요.
    1. 이 정보가 충분한지 여부를 "충분함", "보완 필요" 중 하나로 판단해주세요.
    2. 만약 "보완 필요"라면, 이 지식을 완성할 수 있는 2~3개의 구체적인 후속 질문(follow-up questions)을 생성해주세요.
    3. 각 질문은 전문적이고 깊이 있는 내용이어야 합니다.
    """
    return prompt.strip()

print("\n" + "="*80)
print("💡 [Part 2] 창의적 실습: 지식 검증 프롬프트 생성기 만들기!")
print("======================================================================================")
print("👋 우리는 데이터를 로우(Raw)하게 보는 대신, '지능적인 질문'을 만들어내고 있습니다. 멋지죠?")

generated_prompts = []
# 샘플링된 데이터 리스트를 순회하며 프롬프트 생성
for i in range(min(5, len(sample_data_list))): # 상위 5개만 예시로 생성
    sample = sample_data_list[i]
    prompt = generate_knowledge_check_prompt(sample['question'], sample['answer'], i + 1)
    generated_prompts.append(prompt)
    
    print(f"\n==================================================")
    print(f"🤖 [샘플 {i+1}번] 프롬프트가 생성되었습니다!")
    print("--------------------------------------------------")
    print(prompt)

print("\n✨ 완성된 프롬프트들을 리스트로 모아봤어요:")
# 최종 결과물을 리스트로 출력하여 구조적 이해를 높입니다.
print("pprint.pprint(generated_prompts)를 사용하면 이 프롬프트들을 한눈에 관리할 수 있답니다!")

print("\n✅ 실습 종료! 축하해요! 🎉")
print("이제 여러분은 단순히 데이터를 읽는 것을 넘어, 그 데이터를 활용하여 AI가 '생각할 수 있는' 프롬프트를 디자인하는 능력을 갖추게 되었어요!")